In [ ]:
# In[ ]:


# I will use in this Kernel the step-by-step process of Will Koehrsen.
# I won't use everything, but most of them.
# This project at in GitHub repository: https://github.com/WillKoehrsen/machine-learning-project-walkthrough

In [ ]:
# In[ ]:


# # Imports

# Pandas and numpy for data manipulation
import pandas as pd
import numpy as np

# No warnings about setting value on copy of slice
pd.options.mode.chained_assignment = None

# Display up to 60 columns of a dataframe
pd.set_option('display.max_columns', 60)

# Matplotlib visualization
import matplotlib.pyplot as plt
%matplotlib inline

# Set default font size
plt.rcParams['font.size'] = 24

# Internal ipython tool for setting figure size
from IPython.core.pylabtools import figsize

# Seaborn for visualization
import seaborn as sns
sns.set(font_scale = 2)

# Splitting data into training and testing
from sklearn.model_selection import train_test_split

In [ ]:
# In[ ]:


# # # Data Cleaning and Formatting

# # Load in the Data and Examine

# Read in data into a dataframe 
data = pd.read_csv('../input/train_v2.csv')

# Display top of dataframe
data.head()

In [ ]:
# In[ ]:


data.shape

In [ ]:
# In[ ]:


# # Data Types and Missing Values

# See the column data types and non-missing values
data.info()

In [ ]:
# In[ ]:


data.select_dtypes(include=['object']).head()

In [ ]:
# In[ ]:


# Statistics for each column
data.describe()

In [ ]:
# In[ ]:


# # Missing Values

# Function to calculate missing values by column
def missing_values_table(df):
        # Total missing values
        mis_val = df.isnull().sum()
        
        # Percentage of missing values
        mis_val_percent = 100 * df.isnull().sum() / len(df)
        
        # Make a table with the results
        mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
        
        # Rename the columns
        mis_val_table_ren_columns = mis_val_table.rename(
        columns = {0 : 'Missing Values', 1 : '% of Total Values'})
        
        # Sort the table by percentage of missing descending
        mis_val_table_ren_columns = mis_val_table_ren_columns[
            mis_val_table_ren_columns.iloc[:,1] != 0].sort_values(
        '% of Total Values', ascending=False).round(1)
        
        # Print some summary information
        print(("Your selected dataframe has " + str(df.shape[1]) + " columns.\n"      
            "There are " + str(mis_val_table_ren_columns.shape[0]) +
              " columns that have missing values."))
        
        # Return the dataframe with missing information
        return mis_val_table_ren_columns

In [ ]:
# In[ ]:


missing_values_table(data).head(50)

In [ ]:
# In[ ]:


# Mean imputation learns from the data, so it is no longer fitted on the whole
# dataset here: it is the SimpleImputer(strategy='mean') step of the pipeline
# built after the train/test split, fitted on the training data only.


In [ ]:
# In[ ]:


missing_values_table(data).head(50)

In [ ]:
# In[ ]:


# Numeric missing values are imputed later (training data only), so only drop
# the rows that still have missing values in the object-dtype columns.
data.dropna(subset=data.select_dtypes(include=['object']).columns, inplace=True)
missing_values_table(data)

In [ ]:
# In[ ]:


data.shape

In [ ]:
# In[ ]:


# # # Exploratory Data Analysis

for i in data.select_dtypes(include=['object']).columns:
    data.drop(labels=i, axis=1, inplace=True)

In [ ]:
# In[ ]:


# # Single Variable Plots

figsize=(8, 8)

# Histogram of the loss
plt.style.use('fivethirtyeight')
plt.hist(data['loss'], bins = 100, edgecolor = 'k')
plt.xlabel('Loss') 
plt.ylabel('Number of Clients');
plt.title('Loss Distribution')

In [ ]:
# In[ ]:


# # Correlations between Features and Target

# Find all correlations and sort 
correlations_data = data.corr()['loss'].sort_values()

# Print the most negative correlations
print((correlations_data.head(15), '\n'))

# Print the most positive correlations
print((correlations_data.tail(15)))

In [ ]:
# In[ ]:


# Dropping constant columns is a data-driven feature selection, so it is no
# longer done on the whole dataset here: it is the VarianceThreshold(threshold=0.0)
# step of the pipeline built after the train/test split (training data only).


In [ ]:
# In[ ]:


# Find all correlations and sort 
correlations_data = data.corr()['loss'].sort_values()

# Print the most negative correlations
print((correlations_data.head(15), '\n'))

# Print the most positive correlations
print((correlations_data.tail(15)))

In [ ]:
# In[ ]:


data.shape

In [ ]:
# In[ ]:


# # # Feature Engineering and Selection

from sklearn.base import BaseEstimator, TransformerMixin

class RemoveCollinearFeatures(BaseEstimator, TransformerMixin):
    '''
    Objective:
        Remove collinear features with a correlation coefficient greater than
        the threshold. Removing collinear features can help a model to
        generalize and improves the interpretability of the model.
        The correlations are computed in fit() (training data only) and the
        same columns are then dropped from any data passed to transform().
    
    Inputs: 
        threshold: any features with correlations greater than this value are removed
    
    Output: 
        array that contains only the non-highly-collinear features
    '''
    def __init__(self, threshold=0.6):
        self.threshold = threshold
    
    def fit(self, x, y=None):
        # The target is not part of x, so correlations with loss are not used
        
        # Calculate the correlation matrix
        corr_matrix = pd.DataFrame(x).corr().values
        iters = list(range(corr_matrix.shape[1] - 1))
        drop_cols = []

        # Iterate through the correlation matrix and compare correlations
        for i in iters:
            for j in range(i):
                val = abs(corr_matrix[j, i + 1])
                
                # If correlation exceeds the threshold
                if val >= self.threshold:
                    drop_cols.append(i + 1)

        # Drop one of each pair of correlated columns
        drops = set(drop_cols)
        self.keep_cols_ = [k for k in range(corr_matrix.shape[1]) if k not in drops]
        return self
    
    def transform(self, x):
        return np.asarray(x)[:, self.keep_cols_]

In [ ]:
# In[ ]:


# Remove the collinear features above a specified correlation coefficient.
# This is now fitted on the training data only: RemoveCollinearFeatures(threshold=0.6)
# is a step of the pipeline built after the train/test split.


In [ ]:
# In[ ]:


data.shape

In [ ]:
# In[ ]:


# # # Split Into Training and Testing Sets

# Separate out the features and targets
features = data.drop(columns='loss')
targets = pd.DataFrame(data['loss'])

# Split into 80% training and 20% testing set
X_train, X_test, y_train, y_test = train_test_split(features, targets, test_size = 0.2, random_state = 42)

print((X_train.shape))
print((X_test.shape))
print((y_train.shape))
print((y_test.shape))

In [ ]:
# In[ ]:


# # Feature Scaling
# Imputation, constant-column removal, collinear-feature removal and scaling are
# all fitted on the training data only. They are chained with each model in a
# Pipeline, so cross-validation also refits them on every training fold.
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

def build_pipeline(model):
    return Pipeline([
        ('impute', SimpleImputer(strategy='mean')),
        ('drop_constant', VarianceThreshold(threshold=0.0)),
        ('remove_collinear', RemoveCollinearFeatures(threshold=0.6)),
        ('scale', StandardScaler()),
        ('model', model),
    ])

In [ ]:
# In[ ]:


# Convert y to one-dimensional array (vector)
y_train = np.array(y_train).reshape((-1, ))
y_test = np.array(y_test).reshape((-1, ))

In [ ]:
# In[ ]:


X_train

In [ ]:
# In[ ]:


X_test

In [ ]:
# In[ ]:


# # # Models to Evaluate

# We will compare five different machine learning Cassification models:

# 1 - Logistic Regression
# 2 - K-Nearest Neighbors Classification
# 3 - Suport Vector Machine
# 4 - Naive Bayes
# 5 - Random Forest Classification

# Function to calculate mean absolute error
def cross_val(X_train, y_train, model):
    # Applying k-Fold Cross Validation
    from sklearn.model_selection import cross_val_score
    accuracies = cross_val_score(estimator = model, X = X_train, y = y_train, cv = 5)
    return accuracies.mean()

# Takes in a model, trains the model, and evaluates the model on the test set
def fit_and_evaluate(model):
    
    # Preprocessing is fitted inside the pipeline (training data / training folds only)
    pipe = build_pipeline(model)
    
    # Train the model
    pipe.fit(X_train, y_train)
    
    # Make predictions and evalute
    model_pred = pipe.predict(X_test)
    model_cross = cross_val(X_train, y_train, pipe)
    
    # Return the performance metric
    return model_cross

In [ ]:
# In[ ]:


# # Naive Bayes
from sklearn.naive_bayes import GaussianNB
naive = GaussianNB()
naive_cross = fit_and_evaluate(naive)

print(('Naive Bayes Performance on the test set: Cross Validation Score = %0.4f' % naive_cross))

In [ ]:
# In[ ]:


# # Random Forest Classification
from sklearn.ensemble import RandomForestClassifier
random = RandomForestClassifier(n_estimators = 10, criterion = 'entropy')
random_cross = fit_and_evaluate(random)

print(('Random Forest Performance on the test set: Cross Validation Score = %0.4f' % random_cross))

In [ ]:
# In[ ]:


# # Gradiente Boosting Classification
from xgboost import XGBClassifier
gb = XGBClassifier()
gb_cross = fit_and_evaluate(gb)

print(('Gradiente Boosting Classification Performance on the test set: Cross Validation Score = %0.4f' % gb_cross))

In [ ]:
# In[ ]:

In [ ]:
# In[ ]:

In [ ]:
# In[ ]: